## Mobile App Usage & User Engagement Analysis
**Industry:** Technology / Mobile Applications

**Business scenario:** Study app sessions, features used, session duration and user ratings to understand engagement and improve the user experience.

**Dataset:** `24_mobile_app_usage.csv` (one row = one app session)

## How to use this notebook
1. Make sure `24_mobile_app_usage.csv` is available (in a `data` folder next to this notebook, or in the same folder).
2. Click **Run > Run All Cells** (Jupyter) or **Runtime > Run all** (Colab). Or press **Shift+Enter** to run one cell at a time.
3. Read the explanation above each result. Each task from the project brief has its own section.

> **Note:** If you are using the practice data supplied with this project, the numbers are random and are not real findings.

## Setup: libraries and file settings
We use **pandas** (tables), **NumPy** (statistics) and **matplotlib** (charts).

If your CSV uses different column names, change the names in quotes below. Nothing else needs editing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 20)

# ---- Column names (edit these if your CSV is different) ----
COL_SESSION  = "Session_ID"
COL_USER     = "User_ID"
COL_DATE     = "Session_Date"
COL_FEATURE  = "Feature_Used"
COL_DEVICE   = "Device_Type"
COL_AGE      = "Age_Group"
COL_DURATION = "Session_Duration_Mins"
COL_RATING   = "User_Rating"

# ---- Find the data file automatically ----
candidates = [Path("data/24_mobile_app_usage.csv"),
              Path("../data/24_mobile_app_usage.csv"),
              Path("24_mobile_app_usage.csv")]
DATA_FILE = next((p for p in candidates if p.exists()), None)
if DATA_FILE is None:
    raise FileNotFoundError("Could not find 24_mobile_app_usage.csv. "
                            "Put it next to this notebook (or in a 'data' folder).")

# ---- Where results are saved ----
OUT_DIR = Path("output") if Path("data").exists() else Path(".")
OUT_DIR.mkdir(exist_ok=True)

print("Using data file:", DATA_FILE)
print("Results will be saved in:", OUT_DIR.resolve())

## Load and inspect
Read the CSV file into a table (a pandas *DataFrame*) and look at its size, column names and data types.

In [ ]:
df = pd.read_csv(DATA_FILE)

needed = [COL_SESSION, COL_USER, COL_DATE, COL_FEATURE, COL_DEVICE,
          COL_AGE, COL_DURATION, COL_RATING]
missing_cols = [c for c in needed if c not in df.columns]
if missing_cols:
    raise KeyError(f"These columns are not in your file: {missing_cols}. "
                   f"Your file has: {list(df.columns)}. Fix the names in the Setup cell.")

print("Rows, columns:", df.shape)
print("Column names:", list(df.columns))

In [ ]:
df.dtypes

In [ ]:
df.head()

## Missing values and duplicates
A **missing value** is a blank cell. A **duplicate** is the same Session_ID appearing more than once. Both can distort results, so we find them first.

In [ ]:
print("Missing values in each column:")
df.isnull().sum()

In [ ]:
dup_count = df.duplicated(subset=COL_SESSION).sum()
print("Duplicate Session_ID rows found:", dup_count)

# Keep the first copy of each Session_ID and drop the repeats
df = df.drop_duplicates(subset=COL_SESSION, keep="first")
print("Rows after removing duplicates:", len(df))

## Handle missing values
**Method chosen:** fill each blank with the **median** of its column.

**Why the median?** A few very long sessions pull the average up, but the median stays close to a typical session. Ratings are whole numbers, so the median also gives a sensible rating.

**Trade-off:** filling with one value for everyone is simple, but it can hide differences between groups.

In [ ]:
df[COL_DURATION] = pd.to_numeric(df[COL_DURATION], errors="coerce")
df[COL_RATING]   = pd.to_numeric(df[COL_RATING], errors="coerce")

df[COL_DURATION] = df[COL_DURATION].fillna(df[COL_DURATION].median())
df[COL_RATING]   = df[COL_RATING].fillna(df[COL_RATING].median())

print("Missing values after cleaning:")
df[[COL_DURATION, COL_RATING]].isnull().sum()

## Datetime features
Dates are read as text at first. Converting them to a real date type lets us pull out the month, day of the week and hour.

In [ ]:
df[COL_DATE] = pd.to_datetime(df[COL_DATE], errors="coerce")
df = df.dropna(subset=[COL_DATE])      # drop rows whose date could not be read

df["Month"]   = df[COL_DATE].dt.month_name()
df["Weekday"] = df[COL_DATE].dt.day_name()
df["Hour"]    = df[COL_DATE].dt.hour

HAS_TIME = df["Hour"].nunique() > 1
if not HAS_TIME:
    print("NOTE: your dates have no time of day, so every Hour is the same.")
    print("Peak-hour results in Task 10 will not be meaningful for this file.")

df[[COL_DATE, "Month", "Weekday", "Hour"]].head()

## Basic metrics
*Unique users* counts each person once, even if they had many sessions.

In [ ]:
total_sessions = df[COL_SESSION].nunique()
unique_users   = df[COL_USER].nunique()
avg_duration   = df[COL_DURATION].mean()

print(f"Total sessions:           {total_sessions}")
print(f"Unique users:             {unique_users}")
print(f"Average session duration: {avg_duration:.2f} minutes")
print(f"Average sessions per user: {total_sessions / unique_users:.1f}")

## Top 10 features
Count how many sessions used each feature and keep the top 10.

In [ ]:
top_features = df[COL_FEATURE].value_counts().head(10)
top_features.to_frame("Sessions")

In [ ]:
plt.figure(figsize=(8, 4.5))
top_features.sort_values().plot(kind="barh", color="#3b6ea5")
plt.title("Top 10 Most Used Features")
plt.xlabel("Number of sessions")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(OUT_DIR / "chart_top_features.png", dpi=150)
plt.show()

## Duration by device type
Do people stay longer on some devices than others?

In [ ]:
device_duration = (df.groupby(COL_DEVICE)[COL_DURATION]
                     .mean().sort_values(ascending=False).round(2))
device_duration.to_frame("Avg duration (mins)")

In [ ]:
plt.figure(figsize=(6, 4))
device_duration.plot(kind="bar", color="#3b6ea5")
plt.title("Average Session Duration by Device")
plt.xlabel("Device type")
plt.ylabel("Minutes")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUT_DIR / "chart_device_duration.png", dpi=150)
plt.show()

## Age groups ranked by sessions

In [ ]:
age_sessions = (df.groupby(COL_AGE)[COL_SESSION].count()
                  .sort_values(ascending=False))
age_sessions.to_frame("Sessions")

## Ratings by feature

In [ ]:
feature_rating = (df.groupby(COL_FEATURE)[COL_RATING]
                    .mean().sort_values(ascending=False).round(2))
feature_rating.to_frame("Avg rating")

## Peak hours and busiest weekdays

In [ ]:
hour_counts = df["Hour"].value_counts().head(5)
print("Top 5 busiest hours (hour of day: number of sessions)")
hour_counts.to_frame("Sessions")

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday",
                 "Friday", "Saturday", "Sunday"]
weekday_counts = df["Weekday"].value_counts()
weekday_counts.to_frame("Sessions")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

weekday_counts.reindex(weekday_order).plot(kind="bar", ax=axes[0], color="#3b6ea5")
axes[0].set_title("Sessions by Weekday")
axes[0].set_xlabel("Weekday")
axes[0].set_ylabel("Number of sessions")
axes[0].tick_params(axis="x", rotation=45)

if HAS_TIME:
    df["Hour"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="#d98c3f")
    axes[1].set_title("Sessions by Hour of Day")
    axes[1].set_xlabel("Hour (0-23)")
    axes[1].set_ylabel("Number of sessions")
else:
    axes[1].text(0.5, 0.5, "No time of day in data", ha="center")
    axes[1].axis("off")

plt.tight_layout()
plt.savefig(OUT_DIR / "chart_peak_usage.png", dpi=150)
plt.show()

## Use a list and a tuple
- A **list** stores the most-used features. Lists can be changed.
- A **tuple** stores the fixed device categories. Tuples cannot be changed after they are created.

In [ ]:
most_used_features = top_features.index.tolist()          # list
device_categories  = tuple(sorted(df[COL_DEVICE].unique()))   # tuple

print("List of most-used features:", most_used_features)
print("Tuple of device categories:", device_categories)

## Use a set and write a function
- A **set** keeps each value only once, so it gives the unique users.
- The **function** `avg_duration_for_feature` returns the average session duration for any feature we give it.

In [ ]:
unique_users_set = set(df[COL_USER])
print("Number of unique users (using a set):", len(unique_users_set))


def avg_duration_for_feature(data, feature):
    """Return the average session duration (minutes) for one feature."""
    subset = data[data[COL_FEATURE] == feature]
    if subset.empty:
        return None
    return subset[COL_DURATION].mean()


for feature in most_used_features[:3]:
    print(f"{feature}: {avg_duration_for_feature(df, feature):.2f} minutes")

## NumPy statistics for session duration
- **Mean**: the average.
- **Median**: the middle value.
- **Standard deviation**: how spread out the durations are. A larger number means more variation.

In [ ]:
durations = df[COL_DURATION].to_numpy()

print(f"Mean:               {np.mean(durations):.2f} minutes")
print(f"Median:             {np.median(durations):.2f} minutes")
print(f"Standard deviation: {np.std(durations):.2f} minutes")

## Unusually long sessions (IQR method) and engagement labels

**IQR method for outliers**
- Q1 = value that 25% of sessions fall below. Q3 = value that 75% fall below.
- IQR = Q3 - Q1.
- A session is unusual if it is shorter than `Q1 - 1.5 x IQR` or longer than `Q3 + 1.5 x IQR`.

**Engagement label rules (documented):** sessions are split into thirds by duration, using the 33rd and 66th percentiles.
- **Low**: duration up to the 33rd percentile
- **Medium**: above the 33rd and up to the 66th percentile
- **High**: above the 66th percentile

In [ ]:
q1, q3 = np.percentile(durations, [25, 75])
iqr = q3 - q1
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

print(f"Q1 = {q1:.2f}, Q3 = {q3:.2f}, IQR = {iqr:.2f}")
print(f"Normal range: {lower_limit:.2f} to {upper_limit:.2f} minutes")

outliers = df[(df[COL_DURATION] < lower_limit) | (df[COL_DURATION] > upper_limit)]
print("Unusual sessions found:", len(outliers))
outliers[[COL_SESSION, COL_USER, COL_FEATURE, COL_DEVICE, COL_DURATION]].head(10)

In [ ]:
p33, p66 = np.percentile(durations, [33, 66])
print(f"Low: up to {p33:.2f} mins | Medium: up to {p66:.2f} mins | High: above {p66:.2f} mins")


def engagement(minutes):
    """Label a session as Low, Medium or High engagement."""
    if minutes <= p33:
        return "Low"
    elif minutes <= p66:
        return "Medium"
    return "High"


df["Engagement"] = df[COL_DURATION].apply(engagement)
engagement_counts = df["Engagement"].value_counts()
engagement_counts.to_frame("Sessions")

In [ ]:
plt.figure(figsize=(6, 4))
engagement_counts.reindex(["Low", "Medium", "High"]).plot(
    kind="bar", color=["#c9d6e3", "#7ea3c7", "#3b6ea5"])
plt.title("Sessions by Engagement Level")
plt.xlabel("Engagement")
plt.ylabel("Number of sessions")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUT_DIR / "chart_engagement.png", dpi=150)
plt.show()

## AppUser class and five recommendations
A **class** is a blueprint for an object. `AppUser` stores a user's ID and device type.

In [ ]:
class AppUser:
    """A simple app user with an ID and a device type."""

    def __init__(self, user_id, device_type):
        self.user_id = user_id
        self.device_type = device_type

    def __repr__(self):
        return f"AppUser(user_id={self.user_id}, device_type={self.device_type})"


first_row = df.iloc[0]
example_user = AppUser(first_row[COL_USER], first_row[COL_DEVICE])
print(example_user)

### Recommendations (generated from the results above)
The numbers below come straight from your data, so they update automatically. Copy them into your report and reword them in your own words.

In [ ]:
top_feature      = top_features.index[0]
low_rated        = feature_rating.index[-1]
best_device      = device_duration.index[0]
worst_device     = device_duration.index[-1]
top_age          = age_sessions.index[0]
busiest_day      = weekday_counts.index[0]
quietest_day     = weekday_counts.index[-1]
high_share       = (df["Engagement"] == "High").mean() * 100

print("1. '%s' is the most used feature (%d sessions). Keep it fast and prominent."
      % (top_feature, top_features.iloc[0]))
print("2. '%s' has the lowest average rating (%.2f). Review its usability and fix pain points."
      % (low_rated, feature_rating.iloc[-1]))
print("3. %s users have the shortest sessions (%.2f mins vs %.2f on %s). Check performance and layout on %s."
      % (worst_device, device_duration.iloc[-1], device_duration.iloc[0], best_device, worst_device))
print("4. The %s age group has the most sessions (%d). Aim new features and campaigns at them."
      % (top_age, age_sessions.iloc[0]))
if HAS_TIME:
    peak_hour = int(hour_counts.index[0])
    print("5. Peak usage is around %d:00 and %s is the busiest day. Send notifications just before "
          "these times, and run re-engagement offers on %s, the quietest day."
          % (peak_hour, busiest_day, quietest_day))
else:
    print("5. %s is the busiest day and %s the quietest. Run re-engagement offers on %s."
          % (busiest_day, quietest_day, quietest_day))
print()
print("Only %.0f%% of sessions are High engagement, so aim to lift this share." % high_share)

## Save results
Save the cleaned data (with the new Month, Weekday, Hour and Engagement columns), the outlier list and a one-line summary.

In [ ]:
df.to_csv(OUT_DIR / "24_cleaned_app_usage.csv", index=False)
outliers.to_csv(OUT_DIR / "24_outliers.csv", index=False)

print("Saved in:", OUT_DIR.resolve())
for name in ["24_cleaned_app_usage.csv", "24_outliers.csv", "chart_top_features.png",
             "chart_device_duration.png", "chart_peak_usage.png", "chart_engagement.png"]:
    print(" -", name)

## Summary and limitations
**Write your conclusion here** in two or three sentences: which feature, device and age group stand out, when users are most active, and your main recommendation.

**Limitations**
- If you used the practice data, the findings are random and are not real insights.
- Filling blanks with the median is a simple choice and can hide differences between groups.
- The results show patterns (associations), not proven causes.